# Bodhan Tool Agent — End-to-End Walkthrough

**Author:** Thomas Antony  
**Repository:** [Bodhan AI Cookbook](https://github.com/Bodhan-AI/Bodhan-AI-Cookbook)  
**Version:** 0.1.0  

---

## Welcome!

This notebook provides a simple, beginner-friendly walkthrough of the **Bodhan Tool Agent**.

### What is the Bodhan Tool Agent?
It is an autonomous AI agent equipped with **Bodhan Indic AI models** that can automatically:
* 📄 **Read documents & images** (PDFs, DOCX, worksheets) with layout-preserving OCR.
* 🎙️ **Transcribe spoken audio** across Indian languages with silence-aware chunking.
* 🌐 **Translate text** accurately between 22 Indian languages and English.
* 🔊 **Synthesize natural speech** in high-fidelity 24kHz audio with authentic regional accents.
* 💰 **Track Rupee costs** in real-time with an automatic budget guard to prevent overspending.

The goal of this walkthrough is to show you **how to get data, how to run the agent, and how to replicate it on your own machine** in just a few minutes — without complex technical jargon.

## Step 1: Environment & API Credentials

To run the agent, make sure your `.env` file contains your **Bodhan API key** and an **LLM API key** (such as OpenAI).

Let's verify which credentials are ready in your environment:

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from src.common.config import BodhanConfig, ensure_utf8_streams

# Ensure proper UTF-8 handling for Indian language scripts
ensure_utf8_streams()
load_dotenv()
config = BodhanConfig.from_env()

print("=== Bodhan Agent Environment Status ===")
print(f"Bodhan Base URL : {config.base_url}")
print(f"LLM Provider    : {config.llm_base_url or 'OpenAI Default'}")
print(f"LLM Model       : {config.llm_model}")
print("\nActive Credentials:")
services = [
    ("Audio Transcription (STT)", bool(config.transcribe_key)),
    ("Text Translation (MT)", bool(config.translate_key)),
    ("Speech Synthesis (TTS)", bool(config.speak_key)),
    ("Document OCR", bool(config.ocr_key)),
    ("Reasoning LLM", bool(config.llm_api_key)),
]
for name, is_configured in services:
    status = "Configured" if is_configured else "Missing (Check .env)"
    print(f"  • {name:<28} : {status}")

## Step 2: Getting the Sample Data

How do we get authentic sample files to test with? We provide a ready-made download script at `src/common/download.py`.

### Option A: From your terminal
```bash
# Download all authentic Indic sample files into sample_data/
python -m src.common.download

# Or fetch only specific files:
python -m src.common.download --keys demo1_pdf demo2_audio

# Or purge downloaded files anytime to reset to a clean slate:
python -m src.common.download --clean
```

### Option B: From Python
You can inspect the curated catalog directly:

In [ ]:
from src.common.download import CATALOG

sample_dir = Path("sample_data")
print("=== Sample Data Catalog ===\n")

for key in ["demo1_pdf", "demo2_audio", "demo3_image", "demo4_clip1_te", "demo5_text", "fail_refusal_video"]:
    spec = CATALOG[key]
    local_file = sample_dir / spec.filename
    if local_file.is_file():
        status = f"Available locally ({local_file.stat().st_size / 1024:.1f} KB)"
    else:
        status = "Run 'python -m src.common.download' to fetch"
    print(f"• [{key}] {spec.filename}")
    print(f"  Purpose: {spec.demo_task}")
    print(f"  Status : {status}\n")

## Step 3: Understanding the Agent's Toolkit

The agent doesn't perform AI generation directly inside the LLM (which could produce hallucinated Indian scripts). Instead, it uses **8 model-facing tools** provided over the Model Context Protocol (MCP):

| Tool Name | Type | What it does | Cost |
|---|---|---|---|
| `extract_document_text` | Bodhan AI | Reads PDFs, DOCX, and images with layout-aware OCR | ₹0.20 / page |
| `transcribe_audio` | Bodhan AI | Converts spoken audio into text | ₹0.10 / min |
| `translate_text` | Bodhan AI | Translates between 22 Indian languages and English | ₹0.20 / 10k tokens |
| `synthesize_speech` | Bodhan AI | Generates natural 24kHz spoken audio | ₹6.00 / 10k chars |
| `detect_language` | Local Helper | Identifies script & language from Unicode characters | **₹0.00 (Free)** |
| `inspect_media` | Local Helper | Reads file metadata (pages, duration, sample rate) | **₹0.00 (Free)** |
| `read_text_file` | Local Helper | Reads text files safely with character limits | **₹0.00 (Free)** |
| `write_text_file` | Local Helper | Saves generated summaries, tables, or subtitles to disk | **₹0.00 (Free)** |

### Quick Demo: Zero-Cost Language Detection
Here is how `detect_language` identifies Indian scripts instantly without spending any API budget:

In [ ]:
import json
from src.bodhan_mcp.server import detect_language

samples = {
    "Hindi"   : "नमस्ते, आप कैसे हैं?",
    "Tamil"   : "வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?",
    "Bengali" : "নমস্কার, আপনি কেমন আছেন?",
    "Telugu"  : "నమస్కారం, మీరు ఎలా ఉన్నారు?",
    "Punjabi" : "ਸਤ ਸ੍ਰੀ ਅਕਾਲ ਜੀ, ਕੀ ਹਾਲ ਹੈ?",
}

print("=== Zero-Cost Language & Script Detection ===")
for name, phrase in samples.items():
    result_json = await detect_language(phrase)
    info = json.loads(result_json)
    print(f"{name:<10} | {info['script']:<12} | Likely: {str(info['likely_languages']):<20} | Confidence: {info['confidence']}")

## Step 4: Budget Guard & Rupee Cost Ledger

Bodhan models bill in **Indian Rupees (₹)**. To ensure the agent never runs wild or overspends, the `CostLedger` monitors spending and acts as a pre-flight circuit breaker.

If an operation would exceed your specified budget (e.g. `max_budget_inr=5.0`), the agent halts cleanly before dispatching the call:

In [ ]:
from src.common.ledger import CostLedger

# Set a strict ₹5.00 budget limit
ledger = CostLedger(max_budget_inr=5.0)

# Record operations in a typical pipeline
ledger.record_operation("indic-ocr", units=4.0, step=1, details="4-page PDF extraction")
ledger.record_operation("indic-translate", units=2000, step=2, details="2,000 output tokens")
ledger.record_operation("indic-speak", units=1500, step=3, details="1,500 chars Tamil audio")

print("=== Rupee Cost Ledger Audit ===")
for r in ledger.history:
    print(f"Step {r['step']} | {r['model']:<16} | Units: {r['units']:>6.1f} | Cost: ₹{r['cost_inr']:>6.4f} | Spent: ₹{r['cumulative_cost_inr']:>6.4f} | {r['details']}")

print("-" * 65)
print(f"Total Spent      : ₹{ledger.cumulative_cost_inr:.4f} / ₹{ledger.max_budget_inr:.2f} limit")
print(f"Remaining Budget : ₹{ledger.remaining_budget_inr:.4f}")
print(f"Can afford 5,000 chars of speech (₹3.00)? {ledger.can_afford('indic-speak', 5000)}")
print(f"Can afford 10,000 chars of speech (₹6.00)? {ledger.can_afford('indic-speak', 10000)} (Circuit breaker prevents overspending)")

## Step 5: How the Agent Solves Tasks (Breaking It Down)

When you give a task to the agent, it follows a simple **5-step reasoning process**:

1. **Inspect Input Files:** Uses zero-cost tools (`inspect_media`, `read_text_file`) to check the document or audio format.
2. **Plan Next Step:** Reasons in English about what needs to be done next.
3. **Check Budget:** Validates against the rupee budget before calling any paid model.
4. **Execute Tool:** Dispatches the tool call to Bodhan models or local utilities.
5. **Verify & Deliver:** Validates generated artifacts, writes a readable markdown report, and returns.

### Running from the Terminal
Any task can be executed in one command:
```bash
python -m src.bodhan_agent \
  --task "Translate this video into French" \
  --files sample_data/fail_unsupported_french_video.webm \
  --out ./out \
  --max-steps 10 \
  --max-cost-inr 5
```

### Running Programmatically in Python
> **Output Cleanliness Guarantee:** We pass `out_dir="./out/walkthrough"` so all generated files, logs, and artifacts are created neatly inside the `out/` folder and **never pollute the root project folder**.

Let's run this test task to see how the agent reasons, verifies language support, and safely refuses without wasting money:

In [ ]:
from src.bodhan_agent.agent import run_task_agent

print("Running Task: 'Translate this video into French'...")

# Note: out_dir points inside './out/walkthrough' keeping the root folder clean!
result = await run_task_agent(
    task="Translate this video into French",
    files_dir="sample_data/fail_unsupported_french_video.webm",
    out_dir="./out/walkthrough",
    max_steps=10,
    max_cost_inr=5.0,
)

print("\n=== Agent Execution Summary ===")
print(f"Status        : {result['status'].upper()}")
print(f"Steps Taken   : {result['total_steps']}")
print(f"Budget Spent  : ₹{result['total_cost_inr']:.4f} (Zero budget wasted on unsupported requests)")
print(f"Output Folder : {result['output_dir']}")
print(f"\nFinal Answer from Agent:\n{result['final_answer']}")

## Step 6: Exploring the Outputs (Inside `out/`)

Every time the agent runs, it creates a clean, predictable output directory inside `out/`:

```
out/walkthrough/
  ├── result.md      # Human-readable final answer and links to artifacts
  ├── report.json    # Machine-readable summary (status, cost, elapsed time)
  ├── trace.jsonl    # Step-by-step audit log of every tool call and decision
  ├── artifacts/     # Generated media files (audio WAVs, SRT subtitles, Markdown tables)
  └── runs/          # Immutable timestamped archive of every individual execution
```

Let's inspect what was generated in `out/walkthrough`:

In [ ]:
out_dir = Path("./out/walkthrough")

if out_dir.exists():
    print("=== Files Generated in out/walkthrough ===")
    for item in sorted(out_dir.iterdir()):
        kind = "folder" if item.is_dir() else f"{item.stat().st_size} bytes"
        print(f"  • {item.name:<18} ({kind})")

    # Read human-friendly result.md
    result_file = out_dir / "result.md"
    if result_file.is_file():
        print("\n=== Preview of result.md ===")
        print(result_file.read_text(encoding="utf-8").strip())

    # Read structured report.json
    report_file = out_dir / "report.json"
    if report_file.is_file():
        report_data = json.loads(report_file.read_text(encoding="utf-8"))
        print("\n=== Preview of report.json ===")
        summary = {
            "task": report_data.get("task"),
            "status": report_data.get("status"),
            "total_steps": report_data.get("total_steps"),
            "total_cost_inr": report_data.get("total_cost_inr"),
            "max_budget_inr": report_data.get("max_budget_inr"),
            "verification": report_data.get("verification"),
            "artifacts": report_data.get("artifacts"),
        }
        print(json.dumps(summary, indent=2))
else:
    print("out/walkthrough folder does not exist yet. Run Step 5 first.")

## Step 7: Ready-to-Run Tasks (Replicate in Your System)

You can easily replicate any of the 6 standard tasks on your own machine. Here are the ready-to-use commands:

### 1. Multi-Page PDF to Tamil Spoken Summary
Ingests a research paper PDF, extracts text, summarizes in English, translates to Tamil, and generates natural Tamil audio:
```bash
python -m src.bodhan_agent \
  --task "Make a two-minute Tamil audio summary of this chapter" \
  --files sample_data/demo1_indic_nlp_paper.pdf \
  --out ./out
```
*Generated Artifacts:* `out/artifacts/tamil_summary.wav`, `out/artifacts/summary_ta.md`

---

### 2. Hindi Voice Note to English Subtitles (SRT)
Ingests a 3.5-minute spoken Hindi recording, chunks silence, transcribes speech, translates to English, and outputs a timestamped `.srt` file:
```bash
python -m src.bodhan_agent \
  --task "Transcribe this Hindi voice note and give me English subtitles" \
  --files sample_data/demo2_hindi_spoken_monologue.ogg \
  --out ./out
```
*Generated Artifacts:* `out/artifacts/subtitles_en.srt`

---

### 3. Marathi Worksheet Image to Markdown Tables
Extracts Devanagari text and tables from an image, preserving table rows and column headers:
```bash
python -m src.bodhan_agent \
  --task "Transcribe this worksheet image into markdown, keeping the tables intact" \
  --files sample_data/demo3_marathi_devanagari_table.png \
  --out ./out
```
*Generated Artifacts:* `out/artifacts/worksheet_marathi.md`

---

### 4. Multilingual Folder to Telugu Audio Filter & Digest
Scans an audio folder, detects languages without LLM calls, isolates Telugu recordings, transcribes them, and outputs an English executive brief:
```bash
python -m src.bodhan_agent \
  --task "Find all Telugu audio clips in this folder, transcribe them, and give me an English digest" \
  --files sample_data/ \
  --out ./out
```
*Generated Artifacts:* `out/artifacts/telugu_digest_en.md`

---

### 5. Bengali Literature Text to High-Fidelity Speech (TTS)
Splits literature verses along Danda (`।`) punctuation and synthesizes natural Bengali speech using a female voice:
```bash
python -m src.bodhan_agent \
  --task "Generate natural Bengali audio for this text using a female voice" \
  --files sample_data/demo5_bengali_gitanjali_verses.txt \
  --out ./out
```
*Generated Artifacts:* `out/artifacts/bengali_speech.wav`

---

## Step 8: Connecting to Claude Desktop / MCP Clients

Because the tools adhere strictly to the open **Model Context Protocol (MCP)**, you can also connect them directly to desktop assistants like **Claude Desktop**.

Add this to your `claude_desktop_config.json`:
```json
{
  "mcpServers": {
    "bodhan_indic": {
      "command": "python",
      "args": ["-m", "src.bodhan_mcp"],
      "cwd": "d:/path/to/examples/bodhan_tool_agent"
    }
  }
}
```

Claude Desktop will automatically discover all 8 model-facing tools, allowing you to converse and process Indian media files directly in your chat interface!

---
### Summary Checklist
- [x] **Data Acquisition:** Sample files downloaded via `python -m src.common.download`.
- [x] **Output Cleanliness:** All outputs generated strictly inside `out/` (e.g. `out/walkthrough`).
- [x] **Rupee Budget Protected:** Real-time pre-flight circuit breaker prevents unexpected spending.
- [x] **22 Indic Languages Supported:** Native transcription, translation, OCR, and speech synthesis.